In [ ]:
# Setup 1
!apt-get install -y imagemagick-6.q16
!pip install Wand

# Setup 2
!pip install -q win2xcur
!apt-get update -qq
!apt-get install -y -qq imagemagick

from google.colab import files
import zipfile
import os
import subprocess
import glob
import re
import shutil

def sanitize_theme(name):
    return re.sub(r'[^\w\s-]', '', name).strip().replace(' ', '_')

def find_cursor_folder(base_path):
    for root, dirs, file_list in os.walk(base_path):
        if "cursors" in dirs:
            return os.path.join(root, "cursors")
        # fallback: if no 'cursors' folder but there are files without extension, use that folder
        elif any('.' not in f and f.lower() not in ['index.theme', 'readme', 'license', 'copying'] for f in file_list):
            return root
    return None

def get_theme_name(root_path):
    # try to read from index.theme
    for f in glob.glob(os.path.join(root_path, "**", "index.theme"), recursive=True):
        with open(f, encoding='utf-8', errors='ignore') as file:
            for line in file:
                if line.startswith("Name="):
                    return line.split("=")[1].strip()
    # fallback: use parent folder name
    return os.path.basename(os.path.dirname(root_path)) if os.path.dirname(root_path) != "work" else "Custom"

def process_zip(zip_path, work_base="work"):
    # Create a unique work directory for this zip
    work_dir = f"{work_base}_{os.path.basename(zip_path).replace('.zip','')}"
    if os.path.exists(work_dir):
        shutil.rmtree(work_dir)
    os.makedirs(work_dir, exist_ok=True)

    # Extract
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(work_dir)

    # Find cursor folder
    cursor_path = find_cursor_folder(work_dir)
    if not cursor_path:
        print(f"❌ No cursor folder found in {zip_path}")
        shutil.rmtree(work_dir)
        return None

    # Get theme name
    theme = get_theme_name(work_dir)
    safe_theme = sanitize_theme(theme)

    # Output directory
    out_dir = "output"
    os.makedirs(out_dir, exist_ok=True)
    # Clean previous output (just to avoid mixing)
    for f in os.listdir(out_dir):
        os.remove(os.path.join(out_dir, f))

    print(f"🔄 Converting {theme} from {os.path.basename(zip_path)}...")
    try:
        subprocess.run(['x2wincurtheme', cursor_path, '-n', theme, '-o', out_dir], check=True, capture_output=True)
    except:
        print("⚠️ Batch failed, trying individual...")
        for f in os.listdir(cursor_path):
            fp = os.path.join(cursor_path, f)
            if os.path.isfile(fp) and f not in ['index.theme', 'README', 'LICENSE']:
                subprocess.run(['x2wincur', fp, '-o', out_dir], capture_output=True)

    # Package if files exist
    out_zip = f"{safe_theme}_Windows.zip"
    if os.listdir(out_dir):
        !cd {out_dir} && zip -r ../{out_zip} .
        if os.path.exists(out_zip):
            print(f"✅ Done! File: {out_zip} ({os.path.getsize(out_zip)//1024} KB)")
            print("📥 Downloading now...")
            files.download(out_zip)
            # Clean up
            os.remove(out_zip)
        else:
            print("❌ Zip creation failed – files are in 'output' folder")
    else:
        print("❌ No cursors converted – check your zip structure")

    # Cleanup work dir
    shutil.rmtree(work_dir)
    return out_zip

# Main loop
print("🖱️  Xcursor to Windows Cursor Converter")
while True:
    print("\n📤 Upload your Xcursor zip file(s) (select multiple if you like):")
    uploaded = files.upload()
    if not uploaded:
        print("No files uploaded. Exiting.")
        break

    # Process each uploaded zip
    for zip_name in uploaded.keys():
        # uploaded[zip_name] is the file content (bytes) – we need to save it to disk
        with open(zip_name, 'wb') as f:
            f.write(uploaded[zip_name])
        process_zip(zip_name)
        os.remove(zip_name)  # clean up the uploaded zip after processing

    # Ask if the user wants to convert another batch
    again = input("\n🔄 Convert another batch? (y/n): ").strip().lower()
    if again != 'y':
        print("👋 All done!")
        break
